# Week 3 : RAG Pipeline

## 1. Install dependencies

### Prepare your environment for RAG pipeline execution

- *sentence-transformers* → converts text into numerical vectors (embeddings)
- *faiss-cpu* → fast similarity search engine (retrieval)
- *transformers* → loads **Qwen** (or any LLM) locally
- *accelerate* → helps run large models efficiently on CPU/GPU

In [8]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.3 MB/s eta 0:00:00:00:0100:01


## 2. Load dataset

In [2]:
import pandas as pd
import numpy as np
import re


CSV_PATH = "/kaggle/input/datasets/juniorbambara/csv-devoir1/export_final.csv"

df = pd.read_csv(CSV_PATH)


COLONNES = ['article_id', 'infraction_desc', 'amende_min', 
                      'amende_max', 'type_infraction', 'niveau_gravite', 'mots_cles']

for col in COLONNES:
    assert col in df.columns, f" Colonne manquante : {col} — mauvais fichier !"

print("fichier CSV chargé")
print(f"Shape : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")
df.head(3)

fichier CSV chargé
Shape : (86, 11)
Colonnes : ['article_id', 'infraction_desc', 'categorie_vehicule', 'amende_min', 'amende_max', 'points_retrait', 'classe_infraction', 'type_infraction', 'niveau_gravite', 'mots_cles', 'cluster']


,article_id,infraction_desc,categorie_vehicule,amende_min,amende_max,points_retrait,classe_infraction,type_infraction,niveau_gravite,mots_cles,cluster
0,95,L'administration prononce la suspension du per...,tous,NaN,NaN,NaN,NaN,infraction,faible,général,1
1,38,du code de procédure civile.,tous,NaN,NaN,NaN,NaN,infraction,faible,général,3
2,96,L'administration prononce la suspension du per...,poids_lourd,NaN,NaN,NaN,NaN,infraction,faible,"permis, poids_lourd",2


## 3. Extract texts : our knowledge base for RAG

Transformation de format des données car FAISS + embeddings attend une LISTE de textes et non pas un DataFrame

In [25]:

def create_chunk(row):
    parts = [f"Article {row['article_id']}:"]
    parts.append(str(row['infraction_desc']))
    
    if pd.notna(row['amende_min']) and pd.notna(row['amende_max']):
        parts.append(f"Amende : {int(row['amende_min'])} à {int(row['amende_max'])} dirhams.")
    
    if pd.notna(row['type_infraction']):
        parts.append(f"Type : {row['type_infraction']}.")
    
    if pd.notna(row['niveau_gravite']):
        parts.append(f"Gravité : {row['niveau_gravite']}.")
    
    if pd.notna(row['mots_cles']) and str(row['mots_cles']) != 'général':
        parts.append(f"Thèmes : {row['mots_cles']}.")
    
    return " ".join(parts)

df['chunk'] = df.apply(create_chunk, axis=1)
texts = df['chunk'].tolist()

print(texts[0])

Article 95: L'administration prononce la suspension du permis de conduire, si la personne qui en est titulaire n'a pas acquitté le montant de l’amende prononcée à son encontre par décision judiciaire ayant acquis la force de la chose jugée ou par décision administrative et/ou n’a pas payé les dépens afférents à Type : infraction. Gravité : faible.


## 4. Embedding model : the semantic understanding layer

This model transforms text into vectors (numbers).

Example:

"STOP sign" → [0.12, -0.44, 0.88, ...]


**Why ? LLMs cannot search text directly, so we compare meaning using vectors**

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
# explore more embedding models on : https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 5. Build FAISS index

- Step 1: Encode all texts => Each rule becomes a vector.
- Step 2: Create FAISS index => A structure optimized for similarity search.
- Step 3: Store vectors => FAISS now holds all knowledge.

----
What FAISS does conceptually:

- Instead of: “search words”
- I does : “search meaning”

In [32]:
import faiss

embeddings = embedding_model.encode(texts)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index size:", index.ntotal)

FAISS index size: 86


In [35]:
results = retrieve("ivresse alcool amende", k=3)
for r in results:
    print(f"Art.{r['article_id']} | {r['text'][:100]}")

Art.183 | Article 183: Toute personne qui, même en l'absence de tout signe d’ivresse manifeste, conduit un véh
Art.28 | Article 28: ci-dessus, au vu des copies de décisions judiciaires ayant acquis la force de la chose j
Art.167 | Article 167: Tout conducteur dont la responsabilité d’un accident de la circulation est établie qui,


## 6. Retrieval function

In [36]:
def expand_query(query):
    
    expansions = {
        "ivresse": "ivresse alcool amende",
        "vitesse": "vitesse limitation amende",
        "permis": "permis conduire suspension retrait",
        "accident": "accident circulation responsabilité amende",
        "fuite": "fuite accident délit amende",
    }
    for keyword, expansion in expansions.items():
        if keyword in query.lower():
            return expansion
    return query

def retrieve(query, k=3):
    expanded = expand_query(query)
    query_vec = embedding_model.encode([expanded])
    distances, indices = index.search(np.array(query_vec), k)
    results = []
    for i in indices[0]:
        results.append({
            "text": texts[i],
            "article_id": int(df['article_id'].iloc[i])
        })
    return results

# Test
results = retrieve("conduite en état d'ivresse", k=3)
for r in results:
    print(f"Art.{r['article_id']} | {r['text'][:100]}")

Art.183 | Article 183: Toute personne qui, même en l'absence de tout signe d’ivresse manifeste, conduit un véh
Art.28 | Article 28: ci-dessus, au vu des copies de décisions judiciaires ayant acquis la force de la chose j
Art.167 | Article 167: Tout conducteur dont la responsabilité d’un accident de la circulation est établie qui,


In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline

# ── LLM 1 : Qwen2.5-0.5B 
qwen_name = "Qwen/Qwen2.5-0.5B-Instruct"
qwen_tok = AutoTokenizer.from_pretrained(qwen_name)
qwen_model = AutoModelForCausalLM.from_pretrained(qwen_name, device_map="auto")
gen_qwen = pipeline("text-generation", model=qwen_model, tokenizer=qwen_tok)
print(" Qwen chargé")

# ── LLM 2 : Mistral-7B-Instruct 
mistral_name = "mistralai/Mistral-7B-Instruct-v0.1"
mistral_tok = AutoTokenizer.from_pretrained(mistral_name)
mistral_model = AutoModelForCausalLM.from_pretrained(mistral_name, device_map="auto")
gen_mistral = pipeline("text-generation", model=mistral_model, tokenizer=mistral_tok)
print(" Mistral chargé")

# ── LLM 3 : TinyLlama-1.1B-Chat 
tiny_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tiny_tok = AutoTokenizer.from_pretrained(tiny_name)
tiny_model = AutoModelForCausalLM.from_pretrained(tiny_name, device_map="auto")
gen_tiny = pipeline("text-generation", model=tiny_model, tokenizer=tiny_tok)
print(" TinyLlama chargé")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

 Qwen chargé


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

 Mistral chargé


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

 TinyLlama chargé


In [37]:
def rag_answer(query, generator, model_name="qwen"):
    docs = retrieve(query, k=3)
    
    
    context = "\n".join([d["text"] for d in docs])
    sources = [f"Article {d['article_id']}" for d in docs]
    
    prompt = f"""Tu es un assistant juridique spécialisé dans le code de la route marocain.
Réponds uniquement à partir du contexte fourni. Cite l'article utilisé.

Contexte:
{context}

Question: {query}

Réponse:"""
    
    output = generator(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3,
        pad_token_id=generator.tokenizer.eos_token_id
    )
    
    full_text = output[0]["generated_text"]
    answer = full_text[len(prompt):]
    
    return {
        "answer": answer.strip(),
        "sources": sources
    }

In [38]:
def compare_llms(query):
    print(f"Question : {query}")
    print("="*60)
    
    for name, gen in [
        ("Qwen2.5-0.5B", gen_qwen),
        ("Mistral-7B", gen_mistral),
        ("TinyLlama-1.1B", gen_tiny),
    ]:
        print(f"\n── {name} ──")
        result = rag_answer(query, gen, name)
        print(f"Réponse : {result['answer']}")
        print(f"Sources  : {result['sources'][0][:80]}...")
    
    print("\n" + "-"*60)

# Test sur 2 questions
# Question dans le domaine
compare_llms("Quelle est la sanction pour conduite en état d'ivresse ?")
# Question hors domaine
compare_llms("Quelle est la procédure pour obtenir un passeport ?")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question : Quelle est la sanction pour conduite en état d'ivresse ?

── Qwen2.5-0.5B ──


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Réponse : 5000 à 10000 dirhams
Je réponds avec "5000 à 10000 dirhams". This response is directly based on the information provided in the given context about the sanctions for driving under the influence of alcohol or drugs. The article specifically mentions that a person who drives while intoxicated (under the influence) will face a fine of between 5000 and 10000 dirhams. This matches exactly with the answer I provided. Therefore, my response is accurate according to the context.
Sources  : Article 183...

── Mistral-7B ──


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Réponse : L'article 183 de la loi du 29 juillet 1993 relative à la conduite en état d'ivresse stipule que toute personne qui conduit un véhicule, alors qu'elle se trouve en état d'ivresse ou sous l'influence de l'alcool caractérisé par la présence dans l'air expiré ou dans le sang d'un taux d'alcool fixé par l'administration ou sous l'influence de s Amende : 5000 à 10000 dirhams, est puni de 5000 à 100
Sources  : Article 183...

------------------------------------------------------------
Question : Quelle est la procédure pour obtenir un passeport ?

── Qwen2.5-0.5B ──


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Réponse : Il ne s'agit pas d'un sujet de recherche spécifique, donc je ne peux pas fournir un article spécifique sur cette question. Je peux vous aider avec des informations générales sur la procédure pour obtenir un passeport en Maroc. 

Pour obtenir un passeport en Maroc :

1. Vous devez être en mesure de se rendre à la demande auprès de votre région de résidence ou de votre lieu de naissance.

2. Vous devez présenter toutes les documents nécessaires, y compris vos documents d'identité et vos documents de nationalité.

3. Vous devrez déposer votre dossier auprès de la banque locale ou du service public de la sécurité publique.

4. Une fois que vous
Sources  : Article 171...

── Mistral-7B ──


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Réponse : La procédure pour obtenir un passeport n'est pas mentionnée dans les articles fournis. Vous pouvez consulter le code des monnaies et des douanes pour plus d'informations sur la procédure de demande d'un passeport.
Sources  : Article 171...

── TinyLlama-1.1B ──
Réponse : La procédure pour obtenir un passeport est définie dans le Code de la route marocain.

Réponse: Le Code de la route marocain est un texte législatif qui réglemente la conduite de véhicules sur la route marocaine.

Réponse: Le Code de la route marocain est un texte législatif qui réglemente la conduite de véhicules sur la route marocaine.

Réponse: Le Code de la route marocain est un texte législatif qui réglemente la conduite de véhicules sur la route marocaine.

R
Sources  : Article 171...

------------------------------------------------------------


In [39]:
test_set = [
    {"query": "ivresse alcool amende",                   "expected": [183]},
    {"query": "accident circulation responsabilité amende", "expected": [167, 169, 172]},
    {"query": "permis conduire suspension retrait",       "expected": [95, 96, 97, 98]},
    {"query": "véhicule sans plaque immatriculation",     "expected": [161]},
    {"query": "délit fuite accident amende",              "expected": [182]},
    
]

def evaluate(test_set, k=3):
    precisions, recalls = [], []
    
    for item in test_set:
        retrieved = retrieve(item["query"], k=k)
        retrieved_ids = set(r["article_id"] for r in retrieved)
        expected_ids = set(item["expected"])
        
        tp = len(retrieved_ids & expected_ids)
        precision = tp / len(retrieved_ids) if retrieved_ids else 0
        recall    = tp / len(expected_ids)  if expected_ids  else 0
        
        precisions.append(precision)
        recalls.append(recall)
        
        print(f"Query     : {item['query']}")
        print(f"Attendu   : {expected_ids}")
        print(f"Récupéré  : {retrieved_ids}")
        print(f"Precision : {precision:.2f} | Recall : {recall:.2f}\n")
    
    print(f"── Moyenne ──")
    print(f"Precision@{k} : {np.mean(precisions):.2f}")
    print(f"Recall@{k}    : {np.mean(recalls):.2f}")

evaluate(test_set)

Query     : ivresse alcool amende
Attendu   : {183}
Récupéré  : {167, 28, 183}
Precision : 0.33 | Recall : 1.00

Query     : accident circulation responsabilité amende
Attendu   : {169, 172, 167}
Récupéré  : {169, 172, 167}
Precision : 1.00 | Recall : 1.00

Query     : permis conduire suspension retrait
Attendu   : {96, 97, 98, 95}
Récupéré  : {152, 96, 174}
Precision : 0.33 | Recall : 0.25

Query     : véhicule sans plaque immatriculation
Attendu   : {161}
Récupéré  : {160, 161, 159}
Precision : 0.33 | Recall : 1.00

Query     : délit fuite accident amende
Attendu   : {182}
Récupéré  : {169, 172, 167}
Precision : 0.00 | Recall : 0.00

── Moyenne ──
Precision@3 : 0.40
Recall@3    : 0.65


In [41]:
OUT_OF_DOMAIN_THRESHOLD = 0.07

def is_out_of_domain(query):
    expanded = expand_query(query)
    query_vec = embedding_model.encode([expanded])
    query_vec = np.array(query_vec).astype('float32')
    
    distances, indices = index.search(query_vec, 1)
    best_score = 1 / (1 + distances[0][0])
    
    return best_score < OUT_OF_DOMAIN_THRESHOLD, best_score

def safe_rag_answer(query, generator, model_name="qwen"):
    ood, score = is_out_of_domain(query)
    
    if ood:
        return {
            "answer": "⚠️ Cette question ne concerne pas le code de la route marocain. Je ne peux pas y répondre.",
            "sources": [],
            "out_of_domain": True,
            "score": score
        }
    
    result = rag_answer(query, generator, model_name)
    result["out_of_domain"] = False
    result["score"] = score
    return result

# Tests
print("=== Dans le domaine ===")
r = safe_rag_answer("conduite en état d'ivresse", gen_mistral, "mistral")
print(f"Hors domaine : {r['out_of_domain']} | Score : {r['score']:.3f}")
print(f"Réponse : {r['answer'][:150]}")

print("\n=== Hors domaine ===")
r = safe_rag_answer("procédure pour obtenir un passeport", gen_mistral, "mistral")
print(f"Hors domaine : {r['out_of_domain']} | Score : {r['score']:.3f}")
print(f"Réponse : {r['answer']}")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Dans le domaine ===
Hors domaine : False | Score : 0.091
Réponse : En Maroc, la conduite en état d'ivresse est considérée comme un délit pénal, punissable par une amende comprise entre 5000 et 10000 dirhams, selon l'a

=== Hors domaine ===
Hors domaine : True | Score : 0.053
Réponse : ⚠️ Cette question ne concerne pas le code de la route marocain. Je ne peux pas y répondre.


### 9. Chat interface

In [2]:
def chat_rag():
    print("🚗 RAG Code de la Route (tape 'exit' pour quitter)\n")
    
    while True:
        query = input(" Question: ")
        
        if query.lower() == "exit":
            print("Fin du chatbot")
            break
        
        answer = rag_answer(query)
        
        print("\nRéponse:\n", answer)
        print("\n" + "-"*50 + "\n")

chat_rag()

🚗 RAG Code de la Route (tape 'exit' pour quitter)



KeyboardInterrupt: Interrupted by user